# Candidate–Job Matching using Similarity Metrics

## AI for Data Analysis – Final Project

This project explores how text analysis and mathematical similarity measures can support automated
candidate-job matching in recruitment scenarios.

## Introduction

Modern recruitment processes often require evaluating large numbers of candidate CVs against specific technical job requirements.

This project investigates whether text-based similarity analysis can effectively identify relevant and non-relevant candidates by analyzing overlap in technologies, skills, experience, and role requirements.

Using TF-IDF vectorization, the project compares two mathematical similarity approaches:

- Cosine Similarity
- Euclidean Distance

The primary objective is to evaluate how these methods can support recruiter-oriented candidate screening, ranking, and preliminary decision-making processes.

## Problem Definition

Given a collection of candidate CVs and job descriptions, the objective of this project is to:

- transform textual information into numerical representations
- calculate similarity scores between candidates and job roles
- rank candidates based on job relevance
- generate recruiter-oriented screening recommendations

The main challenge is determining whether text-based similarity measures can effectively approximate recruitment relevance and support early-stage candidate screening processes.

## Data Sources

This project uses two independent datasets:

1. Candidate CV profiles
2. Job description profiles

The candidate dataset contains synthetic CV data based on realistic software engineering and technical roles. The job description dataset simulates real-world recruitment requirements across multiple technology domains.

Both datasets were created specifically for similarity analysis and candidate-job matching experiments.

## Technologies Used

The project was developed using the following technologies and libraries:

- Python – main programming language used for data processing and analysis
- pandas – data loading, cleaning, and manipulation
- scikit-learn – TF-IDF vectorization and similarity calculations
- matplotlib – data visualization and graphical analysis
- Jupyter Notebook – interactive project development and documentation environment

## Methodology

The project follows the following workflow:

1. Load and preprocess textual candidate and job description data
2. Clean and normalize the text information
3. Transform textual data into TF-IDF vector representations
4. Calculate similarity scores between candidate profiles and job roles
5. Compare cosine similarity and Euclidean distance approaches
6. Visualize and analyze the similarity results
7. Generate recruiter-oriented screening recommendations based on similarity thresholds

## Mathematical Background

### TF-IDF Vectorization

TF-IDF (Term Frequency – Inverse Document Frequency) is used to transform text documents into numerical feature vectors.

It is defined as:

$$
TF\text{-}IDF(t,d)=TF(t,d)\times\log\left(\frac{N}{DF(t)}\right)
$$

where:

- TF(t,d) is the frequency of term t in document d
- DF(t) is the number of documents containing the term
- N is the total number of documents

TF-IDF gives higher importance to terms that are frequent in a specific document but rare across the dataset.

### Cosine Similarity

Cosine similarity is commonly used in text similarity analysis to measure the similarity between vectorized documents.
It is defined as:

$$
\cos(\theta) =
\frac{A \cdot B}
{\|A\| \|B\|}
$$

where:
- \(A\) and \(B\) are vector representations of text documents
- the numerator represents the dot product
- the denominator normalizes vector magnitude

Higher cosine similarity values indicate stronger similarity between candidate profiles and job descriptions.

### Euclidean Distance

Euclidean distance measures the geometric distance between vectors in multidimensional space.

It is defined as:

$$
d(p,q)=\sqrt{\sum_{i=1}^{n}(p_i-q_i)^2}
$$

Lower Euclidean distance values indicate stronger similarity between candidate and job vectors in multidimensional space.

---

In [3]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import matplotlib.pyplot as plt

In [4]:
cv_df = pd.read_csv("cv_data.csv", sep=";")
job_df = pd.read_csv("job_data.csv", sep=";")

In [5]:
cv_df

,id,name,text
0,1,Alexander Petrov,Senior .NET Developer with 7 years experience ...
1,2,Georgi Ivanov,Junior .NET Developer with 1.5 years experienc...
2,3,Nikolay Dimitrov,"Python Developer with Django, FastAPI, Machine..."
3,4,Martina Petrova,Mid .NET Developer with 4 years experience in ...
4,5,Petar Dimitrov,"Java Backend Developer with Spring Boot, Micro..."
5,6,Elena Nikolova,"Frontend Developer with React, TypeScript, Jav..."
6,7,Viktor Iliev,"DevOps Engineer with Docker, Kubernetes, Azure..."
7,8,Maria Georgieva,"QA Automation Engineer with Selenium, Cypress,..."
8,9,Desislava Ivanova,"Product Owner with Agile, Scrum, Jira, stakeho..."
9,10,Kaloyan Vasilev,"Data Analyst with Python, SQL, Power BI, panda..."


In [6]:
job_df

,id,role,text
0,1,Senior .NET Developer,Looking for Senior .NET Developer with 5+ year...
1,2,Python Developer,Looking for Python Developer with 3+ years exp...
2,3,Java Developer,Looking for a Java Backend Developer with expe...
3,4,Mobile Developer,Seeking a Mobile Developer specialized in Andr...
4,5,Cybersecurity Specialist,We are hiring a Cybersecurity Specialist with ...
5,6,Project Manager,Looking for a Project Manager experienced in A...
6,7,Data Scientist,Seeking a Data Scientist with experience in Py...
7,8,QA Automation Engineer,Looking for a QA Automation Engineer experienc...
8,9,Frontend Developer,"Seeking a Frontend Developer skilled in React,..."


### Dataset Overview
The datasets contain candidate CVs and job descriptions used for similarity analysis. 
The CV dataset includes candidate names and profile descriptions, while the job dataset contains role titles and job requirement texts.

In [7]:
cv_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      20 non-null     int64 
 1   name    20 non-null     object
 2   text    20 non-null     object
dtypes: int64(1), object(2)
memory usage: 612.0+ bytes


In [8]:
job_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      9 non-null      int64 
 1   role    9 non-null      object
 2   text    9 non-null      object
dtypes: int64(1), object(2)
memory usage: 348.0+ bytes


### Missing Values Check

In [10]:
print(cv_df.isnull().sum())
print(job_df.isnull().sum())

id      0
name    0
text    0
dtype: int64
id      0
role    0
text    0
dtype: int64


### Duplicate Records Check

In [11]:
print(cv_df.duplicated().sum())
print(job_df.duplicated().sum())

0
0


### Data Cleaning and Preparation

In [12]:
def preprocess_text(text):
    text = text.lower()
    text = text.replace(".net", "dotnet")
    text = re.sub(r'[^\w\s.]', '', text)
    return text

In [13]:
cv_df["cleaned_text"] = cv_df["text"].apply(preprocess_text)
job_df["cleaned_text"] = job_df["text"].apply(preprocess_text)

In [14]:
cv_df[["text", "cleaned_text"]].head()

,text,cleaned_text
0,Senior .NET Developer with 7 years experience ...,senior dotnet developer with 7 years experienc...
1,Junior .NET Developer with 1.5 years experienc...,junior dotnet developer with 1.5 years experie...
2,"Python Developer with Django, FastAPI, Machine...",python developer with django fastapi machine l...
3,Mid .NET Developer with 4 years experience in ...,mid dotnet developer with 4 years experience i...
4,"Java Backend Developer with Spring Boot, Micro...",java backend developer with spring boot micros...


In [15]:
job_df[["text", "cleaned_text"]].head()

,text,cleaned_text
0,Looking for Senior .NET Developer with 5+ year...,looking for senior dotnet developer with 5 yea...
1,Looking for Python Developer with 3+ years exp...,looking for python developer with 3 years expe...
2,Looking for a Java Backend Developer with expe...,looking for a java backend developer with expe...
3,Seeking a Mobile Developer specialized in Andr...,seeking a mobile developer specialized in andr...
4,We are hiring a Cybersecurity Specialist with ...,we are hiring a cybersecurity specialist with ...


### Prepare Text Data

In [16]:
cv_texts = cv_df["cleaned_text"].tolist()
job_texts = job_df["cleaned_text"].tolist()

In [17]:
cv_texts[:2]

['senior dotnet developer with 7 years experience in c aspdotnet core microservices azure docker and sql server',
 'junior dotnet developer with 1.5 years experience in c aspdotnet core and sql server']

---